In [1]:
import pandas as pd
import numpy as np
import re

from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, RobustScaler, TargetEncoder, FunctionTransformer, LabelEncoder
from sklearn.model_selection import KFold

from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier


from catboost import CatBoostClassifier
import lightgbm as lgb
import xgboost as xgb

import sklearn
sklearn.set_config(transform_output="pandas")
df = pd.read_csv('rps_data.csv', sep=',')

In [2]:
df['score_diff'] = df['score_me_before'] - df['score_opp_before']
df['is_last_round'] = ((df['score_me_before'] == 2) | (df['score_opp_before'] == 2)).astype(int)

In [3]:
df

,match_id,round,player_name,win_category,opp_match_wins,opp_match_winrate,stake,opp_move,my_move,is_last_round,...,score_me_before,score_opp_before,streak_draws,prev_opp_move,prev_my_move,prev_outcome,prev2_opp_move,prev2_my_move,prev2_outcome,score_diff
0,1,1,Неизвестен,unknown,-1,-0.01,25,Н,Б,0,...,0,0,0,-1,-1,none,-1,-1,none,0
1,1,2,Неизвестен,unknown,-1,-0.01,25,К,Н,0,...,0,1,0,Н,Б,lose,-1,-1,none,-1
2,1,3,Неизвестен,unknown,-1,-0.01,25,Н,Н,1,...,0,2,0,К,Н,lose,Н,Б,lose,-2
3,1,4,Неизвестен,unknown,-1,-0.01,25,Н,Н,1,...,0,2,1,Н,Н,draw,К,Н,lose,-2
4,1,5,Неизвестен,unknown,-1,-0.01,25,Н,К,1,...,0,2,2,Н,Н,draw,Н,Н,draw,-2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1200,188,7,Дальний Тайник,<=5,3,0.75,100,Н,Б,1,...,1,2,2,Б,Б,draw,Н,Н,draw,-1
1201,189,1,Дальний Тайник,<=5,4,0.80,50,Н,Б,0,...,0,0,0,-1,-1,none,-1,-1,none,0
1202,189,2,Дальний Тайник,<=5,4,0.80,50,К,Н,0,...,0,1,0,Н,Б,lose,-1,-1,none,-1
1203,189,3,Дальний Тайник,<=5,4,0.80,50,К,К,1,...,0,2,0,К,Н,lose,Н,Б,lose,-2


In [4]:
df['next_opp_move'] = df.groupby('match_id')['opp_move'].shift(-1)
df_model = df[df['round'] >= 3].copy()
df_model = df_model.dropna(subset=['next_opp_move', 'prev_opp_move', 'prev2_opp_move'])

le = LabelEncoder()
X, y = df_model.drop('next_opp_move', axis=1), le.fit_transform(df_model['next_opp_move'])
X_train, X_valid, y_train, y_valid = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

In [5]:
df_model

,match_id,round,player_name,win_category,opp_match_wins,opp_match_winrate,stake,opp_move,my_move,is_last_round,...,score_opp_before,streak_draws,prev_opp_move,prev_my_move,prev_outcome,prev2_opp_move,prev2_my_move,prev2_outcome,score_diff,next_opp_move
2,1,3,Неизвестен,unknown,-1,-0.01,25,Н,Н,1,...,2,0,К,Н,lose,Н,Б,lose,-2,Н
3,1,4,Неизвестен,unknown,-1,-0.01,25,Н,Н,1,...,2,1,Н,Н,draw,К,Н,lose,-2,Н
4,1,5,Неизвестен,unknown,-1,-0.01,25,Н,К,1,...,2,2,Н,Н,draw,Н,Н,draw,-2,Н
8,2,3,Неизвестен,unknown,-1,-0.01,25,Б,Н,0,...,0,1,Б,Б,draw,Н,К,win,1,Б
9,2,4,Неизвестен,unknown,-1,-0.01,25,Б,К,1,...,0,0,Б,Н,win,Б,Б,draw,2,Н
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1196,188,3,Дальний Тайник,<=5,3,0.75,100,Н,К,0,...,1,1,Б,Б,draw,Б,К,lose,-1,К
1197,188,4,Дальний Тайник,<=5,3,0.75,100,К,Н,0,...,1,0,Н,К,win,Б,Б,draw,0,Н
1198,188,5,Дальний Тайник,<=5,3,0.75,100,Н,Н,1,...,2,0,К,Н,lose,Н,К,win,-1,Б
1199,188,6,Дальний Тайник,<=5,3,0.75,100,Б,Б,1,...,2,1,Н,Н,draw,К,Н,lose,-1,Н


In [6]:
drop_features = ["match_id", "player_name","round"]
cat_features = ["win_category", "stake", "my_move", "outcome", "prev_opp_move", "prev_outcome", "prev2_opp_move", "prev2_outcome", "opp_move"]
num_features = ["opp_match_wins", "opp_match_winrate", "score_me_before", "score_opp_before", "streak_draws", "score_diff"]

In [7]:
def clean_column_names(df):
    df.columns = [re.sub(r'[\[\]<>]', '_', col) for col in df.columns]
    return df

low_cardinality = []   
high_cardinality = []       

for col in cat_features:
    n_unique = X_train[col].nunique()
    if n_unique <= 10:
        low_cardinality.append(col)
    else:
        high_cardinality.append(col)

imputer_scaler_encoder = ColumnTransformer(
    transformers = [
        ('drop_features', 'drop', drop_features),
        ('num', Pipeline([
            ('imputer', SimpleImputer(strategy='median')),
            ('scaler', RobustScaler())
        ]), num_features),
        
        ('low_cat', Pipeline([
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
        ]), low_cardinality),
        
        ('high_cat', Pipeline([
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('encoder', TargetEncoder(smooth=10))
        ]), high_cardinality)
    ],
    verbose_feature_names_out=False,
    remainder='drop'
)

preprocessor = Pipeline(
    [
        ('imputer_scaler_encoder', imputer_scaler_encoder),
        ('cleaner', FunctionTransformer(clean_column_names))
    ]
)

In [8]:
results = {
    'train_acc': [],
    'valid_acc': [],
    # 'train_roc_auc': [],
    # 'valid_roc_auc': [],
}
KFoldMean = {
    'mean_acc': [],
    # 'mean_roc_auc': [],
}

index_name = []

def resultTrainValid():
    results_df = pd.DataFrame(
        results,
        index_name
    )
    return results_df

def resultKFoldMean():
    results_KFoldMean = pd.DataFrame(
    KFoldMean,
    index_name
    )
    return results_KFoldMean


def fit_and_add(pipeline, x = X_train, y = y_train):
    pipeline.fit(x, y)
    train_acc_score = accuracy_score(y, pipeline.predict(x))
    valid_acc_score = accuracy_score(y_valid, pipeline.predict(X_valid))
    results['train_acc'].append(train_acc_score)
    results['valid_acc'].append(valid_acc_score)
    # train_roc_auc_score = roc_auc_score(y, pipeline.predict_proba(x)[:, 1])
    # valid_roc_auc_score = roc_auc_score(y_valid, pipeline.predict_proba(X_valid)[:, 1])
    # results['train_roc_auc'].append(train_roc_auc_score)
    # results['valid_roc_auc'].append(valid_roc_auc_score)


def fit_and_add_and_KFoldmean3_for_boost(pipeline, name):
    index_name.append(name)
    train_acc_score = accuracy_score(y_train, pipeline.predict(preprocessor.transform(X_train)))
    valid_acc_score = accuracy_score(y_valid, pipeline.predict(preprocessor.transform(X_valid)))
    results['train_acc'].append(train_acc_score)
    results['valid_acc'].append(valid_acc_score)
    # train_roc_auc_score = roc_auc_score(y_train, pipeline.predict_proba(preprocessor.transform(X_train))[:, 1])
    # valid_roc_auc_score = roc_auc_score(y_valid, pipeline.predict_proba(preprocessor.transform(X_valid))[:, 1])
    # results['train_roc_auc'].append(train_roc_auc_score)
    # results['valid_roc_auc'].append(valid_roc_auc_score)
    cv = KFold(n_splits=3, random_state=42, shuffle=True)
    
    full_pipeline = Pipeline([
        ('preprocessor', preprocessor),
        ('model', pipeline.__class__(**{k:v for k,v in pipeline.get_params().items() 
                                      if k != 'early_stopping_rounds'}))
    ])
    
    cross_acc = cross_val_score(full_pipeline, X, y, cv = cv, scoring='accuracy')
    KFoldMean['mean_acc'].append(cross_acc.mean())
    # cross_roc_auc = cross_val_score(full_pipeline, X, y, cv = cv, scoring='roc_auc')
    # KFoldMean['mean_roc_auc'].append(cross_roc_auc.mean())


def KFoldmean3(pipeline):
    cv = KFold(n_splits=5, random_state=42, shuffle=True)
    
    cross_acc_validation_result = cross_val_score(pipeline, X, y, cv = cv, scoring='accuracy')
    KFoldMean['mean_acc'].append(cross_acc_validation_result.mean())
    # cross_roc_auc_validation_result = cross_val_score(pipeline, X, y, cv = cv, scoring='roc_auc')
    # KFoldMean['mean_roc_auc'].append(cross_roc_auc_validation_result.mean())
    

def create_pipeline(model, name):
    index_name.append(name)
    pipeline = Pipeline(
    [
        ('preprocessor', preprocessor),
        ('model', model)
    ]
    )
    fit_and_add(pipeline)
    KFoldmean3(pipeline)

def create_tree_pipeline_with_rf(name):
    rf = RandomForestClassifier(n_estimators=160, max_depth=8, min_samples_split=3, min_samples_leaf=1, max_features = 'sqrt', random_state=42)
    create_pipeline(rf, name)


In [9]:
preprocessor.fit(X_train, y_train)
preprocessed_X_train = preprocessor.transform(X_train)
preprocessed_X_valid = preprocessor.transform(X_valid)

In [10]:
lr = LogisticRegression(solver = 'saga', C = 15.632429914797136, max_iter = 2688, penalty='elasticnet', l1_ratio = 0.13128978495943688)
knn = KNeighborsClassifier(n_neighbors=18, p=1, weights='distance')
tree_pipeline = DecisionTreeClassifier(max_depth=4, criterion='entropy', min_samples_split=4, min_samples_leaf = 6, random_state=42)

In [11]:
create_pipeline(lr, 'LogReg')
create_pipeline(knn, 'KNN')
create_pipeline(tree_pipeline, 'Tree')
create_tree_pipeline_with_rf('Tree with rf')

f:\Games\КНБ\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
f:\Games\КНБ\.venv\Lib\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
f:\Games\КНБ\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
f:\Games\КНБ\.venv\Lib\site-packages\sklearn\l

In [20]:
cb1 = CatBoostClassifier(
    iterations=500,
    learning_rate=0.07124576882391896,
    depth=3,
    l2_leaf_reg=0.21299532752549347,
    random_strength=0.015967641801089134,
    colsample_bylevel = 0.9615823541984749,
    border_count=240,
    eval_metric='Accuracy',
    min_data_in_leaf = 6
)

cb1.fit(
    preprocessed_X_train,
    y_train,
    eval_set=(preprocessed_X_valid, y_valid),
    # plot=True
)

fit_and_add_and_KFoldmean3_for_boost(cb1, 'CatBoostClassifier1')

0:	learn: 0.4411765	test: 0.3125000	best: 0.3125000 (0)	total: 652us	remaining: 326ms
1:	learn: 0.4352941	test: 0.3281250	best: 0.3281250 (1)	total: 1.15ms	remaining: 286ms
2:	learn: 0.4666667	test: 0.3593750	best: 0.3593750 (2)	total: 1.63ms	remaining: 269ms
3:	learn: 0.4784314	test: 0.3281250	best: 0.3593750 (2)	total: 2.09ms	remaining: 259ms
4:	learn: 0.4686275	test: 0.3203125	best: 0.3593750 (2)	total: 2.53ms	remaining: 251ms
5:	learn: 0.4843137	test: 0.3437500	best: 0.3593750 (2)	total: 2.98ms	remaining: 245ms
6:	learn: 0.4784314	test: 0.3515625	best: 0.3593750 (2)	total: 3.42ms	remaining: 241ms
7:	learn: 0.4784314	test: 0.3515625	best: 0.3593750 (2)	total: 4.02ms	remaining: 247ms
8:	learn: 0.4764706	test: 0.3515625	best: 0.3593750 (2)	total: 4.49ms	remaining: 245ms
9:	learn: 0.4882353	test: 0.3671875	best: 0.3671875 (9)	total: 4.92ms	remaining: 241ms
10:	learn: 0.4921569	test: 0.3750000	best: 0.3750000 (10)	total: 5.36ms	remaining: 238ms
11:	learn: 0.4745098	test: 0.3437500	best:

In [21]:
cb2 = CatBoostClassifier(
    iterations=750,
    learning_rate=0.08564718921836138,
    depth=3,
    l2_leaf_reg=0.35913779146836167,
    bootstrap_type='Bernoulli',
    subsample=0.9779265440256093,
    random_strength=0.015967641801089134,
    border_count=32,
    eval_metric='Accuracy'
)

cb2.fit(
    preprocessed_X_train,
    y_train,
    eval_set=(preprocessed_X_valid, y_valid),
    # plot=True
)

fit_and_add_and_KFoldmean3_for_boost(cb2, 'CatBoostClassifier2')

0:	learn: 0.4450980	test: 0.4453125	best: 0.4453125 (0)	total: 655us	remaining: 491ms
1:	learn: 0.4725490	test: 0.3437500	best: 0.4453125 (0)	total: 1.2ms	remaining: 450ms
2:	learn: 0.4607843	test: 0.3359375	best: 0.4453125 (0)	total: 1.75ms	remaining: 435ms
3:	learn: 0.4666667	test: 0.3359375	best: 0.4453125 (0)	total: 2.51ms	remaining: 468ms
4:	learn: 0.4686275	test: 0.3437500	best: 0.4453125 (0)	total: 3.07ms	remaining: 458ms
5:	learn: 0.4862745	test: 0.3359375	best: 0.4453125 (0)	total: 3.64ms	remaining: 451ms
6:	learn: 0.4843137	test: 0.3437500	best: 0.4453125 (0)	total: 4.08ms	remaining: 433ms
7:	learn: 0.4803922	test: 0.3515625	best: 0.4453125 (0)	total: 4.52ms	remaining: 420ms
8:	learn: 0.4882353	test: 0.3593750	best: 0.4453125 (0)	total: 4.99ms	remaining: 411ms
9:	learn: 0.5019608	test: 0.3437500	best: 0.4453125 (0)	total: 5.42ms	remaining: 401ms
10:	learn: 0.5019608	test: 0.3281250	best: 0.4453125 (0)	total: 6.1ms	remaining: 410ms
11:	learn: 0.5000000	test: 0.3671875	best: 0.

In [14]:
cb3 = CatBoostClassifier(
    iterations=200,
    learning_rate=0.1,
    depth=3,
    l2_leaf_reg=3,
    bagging_temperature=0.5,
    random_strength=1,
    eval_metric='Accuracy',
    early_stopping_rounds=20,
    verbose=50,
    random_state=42
)

cb3.fit(
    preprocessed_X_train, y_train,
    eval_set=(preprocessed_X_valid, y_valid),
    # plot=True
)

fit_and_add_and_KFoldmean3_for_boost(cb3, 'CatBoostClassifier3')

0:	learn: 0.4176471	test: 0.3593750	best: 0.3593750 (0)	total: 784us	remaining: 156ms
Stopped by overfitting detector  (20 iterations wait)

bestTest = 0.40625
bestIteration = 19

Shrink model to first 20 iterations.
0:	learn: 0.4494118	total: 501us	remaining: 99.7ms
50:	learn: 0.5552941	total: 21.7ms	remaining: 63.5ms
100:	learn: 0.6470588	total: 42.5ms	remaining: 41.7ms
150:	learn: 0.6941176	total: 63ms	remaining: 20.5ms
199:	learn: 0.7458824	total: 84.1ms	remaining: 0us
0:	learn: 0.4235294	total: 484us	remaining: 96.4ms
50:	learn: 0.5458824	total: 21.4ms	remaining: 62.5ms
100:	learn: 0.6305882	total: 45ms	remaining: 44.1ms
150:	learn: 0.6800000	total: 70.4ms	remaining: 22.8ms
199:	learn: 0.7176471	total: 98.9ms	remaining: 0us
0:	learn: 0.4131455	total: 727us	remaining: 145ms
50:	learn: 0.5539906	total: 22.6ms	remaining: 66.1ms
100:	learn: 0.6502347	total: 44.6ms	remaining: 43.7ms
150:	learn: 0.6924883	total: 66.4ms	remaining: 21.6ms
199:	learn: 0.7464789	total: 89.6ms	remaining: 0us

In [15]:
# import optuna
# from optuna.samplers import TPESampler
# from catboost import CatBoostClassifier
# from sklearn.model_selection import GroupKFold, GroupShuffleSplit
# from sklearn.metrics import accuracy_score
# from sklearn.preprocessing import LabelEncoder
# import pandas as pd
# import numpy as np

# # ----------------------------
# # 1. Подготовка данных (предполагается, что df_model уже создан)
# # ----------------------------
# exclude_cols = ['match_id', 'player_name', 'next_opp_move']
# X = df_model.drop(columns=exclude_cols, errors='ignore')
# y = LabelEncoder().fit_transform(df_model['next_opp_move'])
# groups = df_model['match_id'].values

# # Список категориальных признаков
# cat_features = [
#     'win_category', 'stake', 'opp_move', 'my_move', 'outcome',
#     'prev_opp_move', 'prev_outcome', 'prev2_opp_move', 'prev2_outcome'
# ]
# cat_features = [col for col in cat_features if col in X.columns]

# # Приводим категориальные колонки к типу 'category'
# for col in cat_features:
#     X[col] = X[col].astype('category')

# # Остальные колонки делаем числовыми
# for col in X.columns:
#     if col not in cat_features:
#         X[col] = pd.to_numeric(X[col], errors='coerce').fillna(0)

# # ----------------------------
# # 2. Функция цели для Optuna
# # ----------------------------
# def objective(trial):
#     params = {
#         'iterations': trial.suggest_int('iterations', 300, 1200, step=50),
#         'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
#         'depth': trial.suggest_int('depth', 3, 10),
#         'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 0.1, 10.0, log=True),
#         'border_count': trial.suggest_int('border_count', 32, 255),
#         'colsample_bylevel': trial.suggest_float('colsample_bylevel', 0.6, 1.0),
#         'min_data_in_leaf': trial.suggest_int('min_data_in_leaf', 1, 50),
#         'random_seed': 42,
#         'verbose': 0,
#         'cat_features': cat_features
#     }
    
#     model = CatBoostClassifier(**params)
#     gkf = GroupKFold(n_splits=5)
#     acc_scores = []
    
#     for train_idx, val_idx in gkf.split(X, y, groups):
#         X_train_fold = X.iloc[train_idx]
#         X_val_fold = X.iloc[val_idx]
#         y_train_fold = y[train_idx]
#         y_val_fold = y[val_idx]
        
#         model.fit(
#             X_train_fold, y_train_fold,
#             eval_set=(X_val_fold, y_val_fold),
#             early_stopping_rounds=50,
#             verbose=False
#         )
#         y_pred = model.predict(X_val_fold)
#         acc = accuracy_score(y_val_fold, y_pred)
#         acc_scores.append(acc)
    
#     return np.mean(acc_scores)

# # ----------------------------
# # 3. Запуск оптимизации
# # ----------------------------
# study = optuna.create_study(direction='maximize', sampler=TPESampler(seed=42))
# study.optimize(objective, n_trials=200, show_progress_bar=True)

# print("Лучшие гиперпараметры:", study.best_params)
# print("Лучшая точность кросс-валидации:", study.best_value)

# # ----------------------------
# # 4. Финальное обучение на валидационном разбиении
# # ----------------------------
# gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
# train_idx, valid_idx = next(gss.split(X, y, groups))
# X_train, X_valid = X.iloc[train_idx], X.iloc[valid_idx]
# y_train, y_valid = y[train_idx], y[valid_idx]

# best_params = study.best_params.copy()
# best_params['random_seed'] = 42
# best_params['verbose'] = 100
# best_params['cat_features'] = cat_features

# final_model = CatBoostClassifier(**best_params)
# final_model.fit(
#     X_train, y_train,
#     eval_set=(X_valid, y_valid),
#     early_stopping_rounds=50,
#     verbose=100
# )

# y_pred_valid = final_model.predict(X_valid)
# valid_acc = accuracy_score(y_valid, y_pred_valid)
# print(f"Точность на валидации: {valid_acc:.4f}")

In [16]:
lgbm = lgb.LGBMClassifier(
    objective='multiclass',       # многоклассовая классификация
    num_class=3,                  # количество классов
    boosting_type='gbdt',
    max_depth=3,
    num_leaves=3,
    subsample_freq=7,
    learning_rate=0.030893841756247098,
    n_estimators=650,
    random_state=42,
    subsample=0.8480028285364561,
    colsample_bytree=0.7726648252169472,
    reg_alpha=1.3973254101399392e-06,
    reg_lambda=0.0004107390806915259,
    min_child_samples=9,
    verbose=-1
)


lgbm.fit(
    preprocessed_X_train,
    y_train,
    eval_set=[(preprocessed_X_valid, y_valid)],
    eval_metric='multi_logloss',
    callbacks=[lgb.early_stopping(stopping_rounds=50),   # ранняя остановка
               lgb.log_evaluation(period=10)]            # логирование каждые 10 итераций
)

fit_and_add_and_KFoldmean3_for_boost(lgbm, 'LGBMClassifier')

Training until validation scores don't improve for 50 rounds
[10]	valid_0's multi_logloss: 1.09185
[20]	valid_0's multi_logloss: 1.09366
[30]	valid_0's multi_logloss: 1.09748
[40]	valid_0's multi_logloss: 1.09314
[50]	valid_0's multi_logloss: 1.09848
Early stopping, best iteration is:
[1]	valid_0's multi_logloss: 1.0885


In [17]:
xgboost = xgb.XGBClassifier(
    objective='multi:softprob',        # многоклассовая классификация (вероятности)
    num_class=3,                       # количество классов
    eval_metric='mlogloss',            # метрика для многоклассовой задачи (или 'merror')
    max_depth=3,
    learning_rate=0.028501619102234265,
    n_estimators=600,
    subsample=0.8877943054742949,
    colsample_bytree=0.6960566694104567,
    colsample_bylevel=0.9460737167069979,
    reg_alpha=0.0002900813074581903,
    reg_lambda=0.001995414273777448,
    gamma=3.296006537275762e-05,
    min_child_weight=4.993110176167802,
    max_delta_step=8.696994123368121,
    random_state=42,
    use_label_encoder=False,
    verbosity=0,
    n_jobs=-1
)

xgboost.fit(
    preprocessed_X_train,
    y_train,
    eval_set=[(preprocessed_X_valid, y_valid)],
    verbose=True
)

fit_and_add_and_KFoldmean3_for_boost(xgboost, 'XGBClassifier')

[0]	validation_0-mlogloss:1.08805
[1]	validation_0-mlogloss:1.08810
[2]	validation_0-mlogloss:1.08826
[3]	validation_0-mlogloss:1.08887
[4]	validation_0-mlogloss:1.08956
[5]	validation_0-mlogloss:1.08938
[6]	validation_0-mlogloss:1.08992
[7]	validation_0-mlogloss:1.09009
[8]	validation_0-mlogloss:1.09127
[9]	validation_0-mlogloss:1.09081
[10]	validation_0-mlogloss:1.09163
[11]	validation_0-mlogloss:1.09211
[12]	validation_0-mlogloss:1.09266
[13]	validation_0-mlogloss:1.09379
[14]	validation_0-mlogloss:1.09382
[15]	validation_0-mlogloss:1.09417
[16]	validation_0-mlogloss:1.09449
[17]	validation_0-mlogloss:1.09553
[18]	validation_0-mlogloss:1.09587
[19]	validation_0-mlogloss:1.09646
[20]	validation_0-mlogloss:1.09711
[21]	validation_0-mlogloss:1.09805
[22]	validation_0-mlogloss:1.09813
[23]	validation_0-mlogloss:1.09881
[24]	validation_0-mlogloss:1.09906
[25]	validation_0-mlogloss:1.09960
[26]	validation_0-mlogloss:1.10037
[27]	validation_0-mlogloss:1.10067
[28]	validation_0-mlogloss:1.1

In [22]:
resultTrainValid()

,train_acc,valid_acc
LogReg,0.431373,0.382812
KNN,0.966667,0.367188
Tree,0.468627,0.296875
Tree with rf,0.886275,0.390625
CatBoostClassifier1,0.827451,0.406250
CatBoostClassifier2,0.698039,0.414062
CatBoostClassifier3,0.458824,0.406250
LGBMClassifier,0.392157,0.390625
XGBClassifier,0.756863,0.359375
CatBoostClassifier1,0.566667,0.414062


In [23]:
resultKFoldMean()

,mean_acc
LogReg,0.355893
KNN,0.358907
Tree,0.384031
Tree with rf,0.341732
CatBoostClassifier1,0.335430
CatBoostClassifier2,0.310420
CatBoostClassifier3,0.347949
LGBMClassifier,0.324512
XGBClassifier,0.326092
CatBoostClassifier1,0.354276


CatBoostClassifier1	0.615686	0.421875
CatBoostClassifier1	0.327627